# Tutorial 4 — Cell Scores

This tutorial computes per-cell disease scores from LDSC enrichment results and
aggregates them by cell-type groupings.

We use the **synthetic plant dataset** from `tests/test_portability.py` — the same
data that drives the portability test. It has:

- 3 000 cells, 12 factors (9 kept)
- 4 tissues, 3 lineages, 8 fine cell types, 3 cultivars
- 2 agronomic traits: `plant_height`, `grain_yield`

No real DRVI model or GWAS data is needed — we build synthetic loadings and results
directly.

In [1]:
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd

from scads_drvi.config import Project
from scads_drvi.io.artifacts import read_loadings, load_interpretation
from scads_drvi.labels import load_labels
from scads_drvi.scores.cell import cs_from_z, ScoreKind
from scads_drvi.scores.aggregate import (
    summarize_by,
    group_matrix,
    block_order,
    eta_squared,
)
from scads_drvi.stats import add_fdr, significant

## 1. Build the synthetic analysis on disk

This replicates the `analysis` fixture from `tests/test_portability.py`.

In [2]:
import h5py

TISSUES = ("leaf", "root", "stem", "flower")
LINEAGES = ("mesophyll", "epidermis", "vasculature")
FINE_TYPES = {
    "mesophyll": ("meso_1", "meso_2", "meso_3"),
    "epidermis": ("epi_1", "epi_2"),
    "vasculature": ("vasc_1", "vasc_2", "vasc_3"),
}
CULTIVARS = ("cv_alpha", "cv_beta", "cv_gamma")
TRAITS = ("plant_height", "grain_yield")
N_CELLS = 3000
N_FACTORS = 12
N_KEPT = 9

root = Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)

fine = [t for lineage in LINEAGES for t in FINE_TYPES[lineage]]
lineage_of = {t: lin for lin in LINEAGES for t in FINE_TYPES[lin]}

cell_ids = np.array([f"plate{i % 5}:bc{i:05d}".encode() for i in range(N_CELLS)])
fine_codes = rng.integers(0, len(fine), size=N_CELLS).astype(np.int32)
tissue_codes = rng.integers(0, len(TISSUES), size=N_CELLS).astype(np.int32)
cultivar_codes = rng.integers(0, len(CULTIVARS), size=N_CELLS).astype(np.int32)
reads = rng.integers(500, 200_000, size=N_CELLS).astype(np.int64)
in_peaks = (reads * rng.uniform(0.2, 0.8, size=N_CELLS)).astype(np.int64)

obs_path = root / "matrix.h5ad"
with h5py.File(obs_path, "w") as fh:
    obs = fh.create_group("obs")
    obs.attrs["_index"] = "cell_uid"
    obs.attrs["encoding-type"] = "dataframe"
    obs.create_dataset("cell_uid", data=cell_ids)
    for name, codes, cats in (
        ("fine_type", fine_codes, fine),
        ("tissue", tissue_codes, TISSUES),
        ("cultivar", cultivar_codes, CULTIVARS),
    ):
        group = obs.create_group(name)
        group.create_dataset("categories", data=np.array([c.encode() for c in cats]))
        group.create_dataset("codes", data=codes)
    obs.create_dataset("total_reads", data=reads)
    lineage_codes = np.array(
        [LINEAGES.index(lineage_of[fine[c]]) for c in fine_codes], dtype=np.int32
    )
    group = obs.create_group("lineage")
    group.create_dataset("categories", data=np.array([l.encode() for l in LINEAGES]))
    group.create_dataset("codes", data=lineage_codes)

proj = Project(root=root, fit="assembly_v3", traits=TRAITS)
fit_dir = proj.fit_dir()
fit_dir.mkdir(parents=True)

dims = [f"dim_{j}" for j in range(N_FACTORS)]
loadings_arr = np.clip(rng.normal(0.4, 0.5, size=(N_CELLS, N_FACTORS)), 0, None)
np.savez(
    fit_dir / "topic_loadings.npz",
    cells=np.array([c.decode() for c in cell_ids], dtype=object),
    factors=np.array(dims),
    loadings=loadings_arr.astype(np.float32),
)

arm = "assembly_v3_topfrac"
arm_dir = proj.enrich_dir(arm)
(arm_dir / "results").mkdir(parents=True)
kept_flags = [True] * N_KEPT + [False] * (N_FACTORS - N_KEPT)
pd.DataFrame({
    "dim": dims,
    "vanished": [False] * N_FACTORS,
    "kept": kept_flags,
    "drop_reason": ["" if k else "annot_too_small" for k in kept_flags],
    "annot_index": [i + 1 if k else None for i, k in enumerate(kept_flags)],
}).to_csv(arm_dir / "factor_map.tsv", sep="\t", index=False)

for trait_index, trait in enumerate(TRAITS):
    trait_dir = arm_dir / "results" / trait
    trait_dir.mkdir(parents=True)
    for slot in range(N_KEPT):
        z = 4.5 - 0.6 * slot + 0.3 * trait_index
        pd.DataFrame({
            "Category": [f"k{slot + 1}L2_0"],
            "Coefficient": [1e-8 * z],
            "Coefficient_std_error": [1e-8],
            "Coefficient_z-score": [z],
            "n_categories": [40],
            "total_h2": [0.11],
        }).to_csv(trait_dir / f"k{slot + 1}.results", sep="\t", index=False)

print("Synthetic dataset ready at:", root)

Synthetic dataset ready at: /tmp/tmpujpgj5us


## 2. Load the interpretation

`load_interpretation` is the one-call entry point: it reads obs, factor labels,
enrichment results (with BH), loadings, and the UMAP embedding.

In [3]:
interp = load_interpretation(
    proj,
    arm,
    obs_path=obs_path,
    obs_columns=["fine_type", "tissue", "cultivar", "lineage", "total_reads"],
)

print("n_cells:  ", interp.n_cells)
print("n_kept:   ", interp.labels.n_kept)
print("traits:   ", interp.traits)
print()
print("Results columns:", interp.results.columns.tolist())

n_cells:   3000
n_kept:    9
traits:    ('plant_height', 'grain_yield')

Results columns: ['Category', 'Coefficient', 'Coefficient_std_error', 'Coefficient_z-score', 'n_categories', 'total_h2', 'trait', 'annot', 'dim', 'display', 'p_1tailed', 'fdr_q']


### Significant factors for `plant_height`

In [4]:
primary = interp.for_trait("plant_height")
sig_mask = significant(primary)
print(f"{sig_mask.sum()} / {len(primary)} factors significant (FDR 5%)")
primary[sig_mask][["dim", "Coefficient_z-score", "fdr_q"]]

5 / 9 factors significant (FDR 5%)


,dim,Coefficient_z-score,fdr_q
0,dim_0,4.5,0.000031
1,dim_1,3.9,0.000216
2,dim_2,3.3,0.001450
3,dim_3,2.7,0.007801
4,dim_4,2.1,0.032156


## 3. Compute per-cell scores with `cs_from_z`

`cs_from_z` computes a z-weighted loading sum: for each cell it multiplies the
per-factor loadings by the z-scores (clipping negatives to zero), then sums.

The `CellScores` object carries its own null value so you never need to hardcode 0
or 1 next to an axis label.

In [5]:
loadings = read_loadings(
    proj.contract(arm)["loadings"],
    npz=proj.contract(arm)["loadings_npz"],
    dims=list(interp.labels.kept_dims),
)

scores = cs_from_z(
    loadings,
    primary,
    model=arm,
    trait="plant_height",
    labels=interp.labels,
)

print("kind:       ", scores.kind)      # ScoreKind.Z_WEIGHTED
print("null value: ", scores.null)      # 0.0 — don't hardcode
print("label:      ", scores.label)     # for axis labels
print("n_unscored: ", scores.n_unscored)
print()
scores.describe()

kind:        ScoreKind.Z_WEIGHTED
null value:  0.0
label:       $CS_i$ (z-weighted loading sum)
n_unscored:  0



count    3000.000000
mean        8.876439
std         3.132355
min         0.873704
25%         6.639145
50%         8.767357
75%        10.887936
max        20.053701
null        0.000000
Name: assembly_v3_topfrac:plant_height, dtype: float64

## 4. Aggregate by grouping variables

`summarize_by` computes mean ± SEM per group, dropping groups with fewer than
`min_cells` cells (and counting the drop).

In [6]:
cells = interp.cells

summary = summarize_by(scores.values, cells, by=["fine_type"], min_cells=20)

print(f"{len(summary)} groups, {summary.n_dropped} dropped (< 20 cells)")
print()
summary.frame.sort_values("mean", ascending=False)

8 groups, 0 dropped (< 20 cells)



,fine_type,n,n_scored,mean,median,std,sem,ci_low,ci_high
5,vasc_1,368,368,9.012564,8.974697,3.293289,0.171675,8.676088,9.349040
6,vasc_2,405,405,9.004446,8.886587,3.037881,0.150954,8.708582,9.300309
2,meso_3,374,374,8.968006,8.789423,3.268878,0.169030,8.636714,9.299298
1,meso_2,356,356,8.930520,8.810499,3.162489,0.167612,8.602008,9.259033
0,meso_1,358,358,8.885566,8.686135,3.063799,0.161927,8.568195,9.202936
7,vasc_3,396,396,8.859520,8.798958,3.128906,0.157233,8.551349,9.167692
3,epi_1,347,347,8.686446,8.608466,2.810505,0.150876,8.390735,8.982157
4,epi_2,396,396,8.659079,8.760607,3.253012,0.163470,8.338684,8.979474


### Multi-level grouping with `block_order`

When you have a hierarchy (lineage > fine type), `block_order` returns the names in
an order that keeps fine types within their lineage block — for use as the `order`
argument to a figure.

In [7]:
blocked = summarize_by(scores.values, cells, by=["lineage", "fine_type"], min_cells=20)
order, blocks = block_order(blocked, group="fine_type", block="lineage")

print("Ordered fine types:", order)
print()
for lineage_name, start, stop in blocks:
    print(f"  {lineage_name}: {order[start:stop]}")

Ordered fine types: ['meso_3', 'meso_2', 'meso_1', 'epi_1', 'epi_2', 'vasc_1', 'vasc_2', 'vasc_3']

  mesophyll: ['meso_3', 'meso_2', 'meso_1']
  epidermis: ['epi_1', 'epi_2']
  vasculature: ['vasc_1', 'vasc_2', 'vasc_3']


## 5. Two-dimensional group matrix

`group_matrix` computes a cell-type × tissue mean matrix — useful for the grouped
landscape figure in Tutorial 5.

In [8]:
means, counts = group_matrix(
    scores.values,
    cells,
    index="fine_type",
    columns="tissue",
    min_cells=10,
    column_order=list(TISSUES),
)

print("means shape:", means.shape)
print()
means.style.background_gradient(cmap="Blues", axis=None)

means shape: (8, 4)



tissue,leaf,root,stem,flower
fine_type,,,,
meso_1,8.759053,8.774409,9.384073,8.708774
meso_2,8.869352,9.061930,8.751322,9.072560
meso_3,8.878075,9.182272,8.805874,9.035039
epi_1,8.582313,8.659037,8.689342,8.812798
epi_2,8.647123,8.596140,8.547949,8.874882
vasc_1,9.024754,8.835208,9.396815,8.890148
vasc_2,9.222542,8.548507,9.052508,9.218083
vasc_3,8.898268,9.274458,8.379518,8.845677


## 6. Measure covariate association with `eta_squared`

`eta_squared` measures how much of the score variance is explained by a categorical
grouping (one-way ANOVA effect size).

In [9]:
for col in ["fine_type", "tissue", "cultivar"]:
    rho2 = eta_squared(
        scores.values.to_numpy(),
        cells[col].astype(str).to_numpy(),
    )
    print(f"  eta² ({col:12s}): {rho2:.4f}")

  eta² (fine_type   ): 0.0017
  eta² (tissue      ): 0.0001
  eta² (cultivar    ): 0.0003
